<a href="https://colab.research.google.com/github/mowumialabi/west-africa-climate-trap-2026/blob/main/era5_thermodynamic_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import ee
import pandas as pd
import numpy as np

# =====================================================================
# CONFIGURATION: DEFINE YOUR GOOGLE CLOUD PROJECT ID HERE
# =====================================================================
# Replace this placeholder string with your actual Google Cloud/GEE Project ID
GEE_PROJECT_ID = 'ee-mowumialabi'

# =====================================================================
# STEP 1: EARTH ENGINE AUTHENTICATION & PROJECT INITIALIZATION
# =====================================================================
print("Initializing Google Earth Engine Connection...")
try:
    ee.Initialize(project=GEE_PROJECT_ID)
    print(f"Earth Engine successfully initialized with project: {GEE_PROJECT_ID}")
except Exception as e:
    print("Authentication/initialization failed. Launching authorization flow...")
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT_ID)
    print(f"Earth Engine successfully authenticated and initialized with project: {GEE_PROJECT_ID}")

# =====================================================================
# STEP 2: DEFINE STUDY AREAS AND TIME FILTER
# =====================================================================
sites = {
    "Ile-Ife": {"lat": 7.484, "lon": 4.484},   # Core Southwest Moisture Trap
    "Abuja":   {"lat": 9.076, "lon": 7.398},   # Mid-Belt Transition Front
    "Kano":    {"lat": 12.002, "lon": 8.592}   # Northern Arid Boundary
}

start_date = "2026-01-01"
end_date = "2026-02-01"

# =====================================================================
# STEP 3: INGEST & FILTER FOR PEAK AFTERNOON METEOROLOGICAL CONDITIONS
# =====================================================================
print(f"Ingesting peak afternoon data columns (12:00 - 16:00 UTC) from {start_date} to {end_date}...")

era5_hourly = (ee.ImageCollection("ECMWF/ERA5_LAND/HOURLY")
               .filterDate(start_date, end_date)
               .filter(ee.Filter.calendarRange(12, 16, 'hour'))
               .select(["temperature_2m", "dewpoint_temperature_2m"]))

mean_climate_layer = era5_hourly.mean()

# =====================================================================
# STEP 4: SPATIAL REDUCTION (PIXEL EXTRACTION)
# =====================================================================
extracted_records = []

print("Extracting regional peak atmospheric variables...")
for name, coords in sites.items():
    point = ee.Geometry.Point([coords["lon"], coords["lat"]])

    stats = mean_climate_layer.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=point,
        scale=9000
    ).getInfo()

    t_air_c = stats["temperature_2m"] - 273.15
    t_dew_c = stats["dewpoint_temperature_2m"] - 273.15

    extracted_records.append({
        "Site": name,
        "Latitude": f"{coords['lat']}°N",
        "Air_Temp_C": t_air_c,
        "Dew_Point_C": t_dew_c
    })

df_gee = pd.DataFrame(extracted_records)

# =====================================================================
# STEP 5: MATHEMATICAL CALCULATIONS (MAGNUS-TETENS FORMULATION)
# =====================================================================
def apply_magnus_tetens_physics(row):
    c1, c2, c3 = 0.61078, 17.27, 237.3

    es = c1 * np.exp((c2 * row["Air_Temp_C"]) / (c3 + row["Air_Temp_C"]))
    ea = c1 * np.exp((c2 * row["Dew_Point_C"]) / (c3 + row["Dew_Point_C"]))

    rh = (ea / es) * 100.0
    vpd = es - ea

    return pd.Series([es, ea, vpd, rh])

print("Applying thermodynamic physics equations...")
df_gee[["es_kPa", "ea_kPa", "VPD_kPa", "RH_Pct"]] = df_gee.apply(apply_magnus_tetens_physics, axis=1)

# =====================================================================
# STEP 6: OUTPUT GENERATION FOR MANUSCRIPT DATA MATRIX
# =====================================================================
print("\n" + "="*65)
print("=== VALIDATED PEAK DIURNAL GEOSPATIAL EXTRACTION TABLE ===")
print("="*65)
print(df_gee.round({
    "Air_Temp_C": 1, "Dew_Point_C": 1,
    "es_kPa": 4, "ea_kPa": 4, "VPD_kPa": 4, "RH_Pct": 1
}).to_string(index=False))


Initializing Google Earth Engine Connection...
Earth Engine successfully initialized with project: ee-mowumialabi
Ingesting peak afternoon data columns (12:00 - 16:00 UTC) from 2026-01-01 to 2026-02-01...
Extracting regional peak atmospheric variables...
Applying thermodynamic physics equations...

=== VALIDATED PEAK DIURNAL GEOSPATIAL EXTRACTION TABLE ===
   Site Latitude  Air_Temp_C  Dew_Point_C  es_kPa  ea_kPa  VPD_kPa  RH_Pct
Ile-Ife  7.484°N        33.7         16.4  5.2374  1.8706   3.3668    35.7
  Abuja  9.076°N        35.1          7.0  5.6599  0.9993   4.6606    17.7
   Kano 12.002°N        31.9         -1.0  4.7205  0.5688   4.1517    12.0
